In [1]:
# Use the shell escape (!) to run the script on Colab's cloud filesystem
!python -c "import torch; print('CUDA Available:', torch.cuda.is_available()); print('GPU Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')"


CUDA Available: True
GPU Device: NVIDIA L4


In [2]:
import os
from google.colab import drive

# 1. Mount Google Drive to access your processed dataset
print("Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Define repository details (Using public HTTPS url)
repo_name = "SME_Credit_Risk"
repo_url = f"https://github.com/mirkosimunovic/{repo_name}.git"

# 3. Clone the public repo (or pull if it already exists)
if not os.path.exists(f"/content/{repo_name}"):
    print(f"\nCloning public repository: {repo_url}...")
    !git clone {repo_url}
else:
    print(f"\nRepository {repo_name} already exists. Pulling latest code changes...")
    %cd /content/{repo_name}
    !git pull
    %cd /content

# 4. Create the target directory inside the cloned repo
!mkdir -p /content/{repo_name}/data/processed

# 5. Copy the cleaned SME dataset from Google Drive
# NOTE: If you saved the CSV inside a specific folder in Google Drive, 
# adjust the source path below (e.g., "/content/drive/MyDrive/YourFolder/processed_sme_final.csv")
drive_source_path = "/content/drive/MyDrive/xAI_Banking_Paper/data/processed/processed_sme_final.csv"
colab_target_path = f"/content/{repo_name}/data/processed/processed_sme_final.csv"

if os.path.exists(drive_source_path):
    !cp "{drive_source_path}" "{colab_target_path}"
    print("\n✓ Cleaned SBA dataset successfully copied from Google Drive to the cloned project")
else:
    print(f"\n⚠️ WARNING: Could not find your dataset at: {drive_source_path}")
    print("Please check your file path inside your Google Drive side panel and update 'drive_source_path'.")

# 6. Change active directory to your repository root
%cd /content/{repo_name}
print(f"\nActive directory set to: {os.getcwd()}")

Mounted at /content/drive

Cloning public repository: https://github.com/mirkosimunovic/SME_Credit_Risk.git...
Cloning into 'SME_Credit_Risk'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 28 (delta 9), reused 24 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 106.84 KiB | 569.00 KiB/s, done.
Resolving deltas: 100% (9/9), done.

✓ Cleaned SBA dataset successfully copied from Google Drive to the cloned project
/content/SME_Credit_Risk

Active directory set to: /content/SME_Credit_Risk


In [31]:
!git pull


Already up to date.


In [ ]:

# 1. Install heavy, CUDA-dependent system libraries first
!pip install "fknni[rapids12]" --extra-index-url=https://pypi.nvidia.com
!pip install faiss-gpu-cu12

# 2. Silently install the rest of our standard project requirements
!pip install -q -r requirements.txt


In [32]:
!python scripts/trainer.py

SME Credit Risk — two-step trainer
Project root: /content/SME_Credit_Risk

Creating output directories if missing:
  [ok] outputs/figures
  [ok] outputs/results
  [ok] models/artifacts
Loading data/processed/processed_sme_final.csv ...
  Rows: 852,308 | Features: 15
  Paid in Full (0): 704,679 | Default (1): 147,629
  Empirical default rate: 0.1732
  Global scale_pos_weight (n0/n1): 4.7733

STEP 1  Stratified 5-Fold Cross-Validation (evaluation only)
Scaler: StandardScaler fit on TRAINING continuous columns of each fold.
Imbalance: native class weights / scale_pos_weight — SMOTE is not used.

Fold 1/5
  Continuous columns scaled (10): ['NAICS', 'Term', 'NoEmp', 'CreateJob', 'RetainedJob', 'FranchiseCode', 'DisbursementGross', 'BalanceGross', 'GrAppv', 'SBA_Appv']
  Left unscaled (binary/low-cardinality): ['RevLineCr_1.0', 'LowDoc_1.0', 'UrbanRural_1.0', 'UrbanRural_2.0', 'NewExist_2.0']
  Train n0/n1 scale_pos_weight = 4.7733
  Fitting Logistic Regression ...
    Logistic Regression  R